# Porto Real Estate: Data Cleaning & Preprocessing Pipeline
This notebook handles the initial data ingestion, geographic filtering, outlier removal, and missing value treatment for the Porto district real estate market. 

The final output will be split into two clean datasets: one for **Apartments** and one for **Houses**; to be used downstream in specialized machine learning notebooks.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [59]:
import pandas as pd

path_to_data = "../data/raw/portugal_listings.csv"
df = pd.read_csv(path_to_data, low_memory=False)
print(f"Raw dataset loaded. Initial shape: {df.shape}")
df.head()

Raw dataset loaded. Initial shape: (135536, 25)


,Price,District,City,Town,Type,EnergyCertificate,GrossArea,TotalArea,Parking,HasParking,...,Elevator,ElectricCarsCharging,TotalRooms,NumberOfBedrooms,NumberOfWC,ConservationStatus,LivingArea,LotSize,BuiltArea,NumberOfBathrooms
0,780000.0,Vila Real,Valpaços,Carrazedo de Montenegro e Curros,Farm,NC,200.0,552450.0,0.0,False,...,False,NaN,NaN,NaN,NaN,NaN,120.0,NaN,NaN,0.0
1,223000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,81.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,81.0,NaN,NaN,2.0
2,228000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,108.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,108.0,NaN,NaN,2.0
3,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,0.0
4,250000.0,Faro,São Brás de Alportel,São Brás de Alportel,Apartment,A+,NaN,114.0,1.0,True,...,True,NaN,2.0,NaN,NaN,NaN,114.0,NaN,NaN,2.0


## 1. Initial Inspection
Before applying structural modifications, we check data types and assess the volume of missing entries per column.

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135536 entries, 0 to 135535
Data columns (total 25 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Price                  135236 non-null  float64
 1   District               135536 non-null  object 
 2   City                   135536 non-null  object 
 3   Town                   135534 non-null  object 
 4   Type                   135520 non-null  object 
 5   EnergyCertificate      135522 non-null  object 
 6   GrossArea              27638 non-null   float64
 7   TotalArea              127153 non-null  float64
 8   Parking                135342 non-null  float64
 9   HasParking             68215 non-null   object 
 10  Floor                  27929 non-null   object 
 11  ConstructionYear       88021 non-null   float64
 12  EnergyEfficiencyLevel  67289 non-null   object 
 13  PublishDate            29239 non-null   object 
 14  Garage                 67289 non-nul

In [19]:
df.isna().mean().sort_values(ascending=False).head(15)

ConservationStatus       0.857661
BuiltArea                0.803617
GrossArea                0.796084
Floor                    0.793937
PublishDate              0.784271
LotSize                  0.707952
NumberOfBedrooms         0.652926
NumberOfWC               0.577559
EnergyEfficiencyLevel    0.503534
ElectricCarsCharging     0.503534
Garage                   0.503534
HasParking               0.496702
TotalRooms               0.459597
ConstructionYear         0.350571
LivingArea               0.225652
dtype: float64

## 2. Core Cleaning & Sanity Rules

We apply the following criteria to ensure data consistency and eliminate anomalies:
* **Geographic Filtering:** Only listings located in the `Porto` district are retained.
* **Property Type:** Restricted exclusively to residential categories: `Apartment` and `House`.
* **Price Bounds:** Addresses the rental/typo anomaly by enforcing a minimum purchase price of €40,000, and caps the maximum price at €1,000,000 to eliminate extreme luxury outliers that distort standard market trends.
* **Construction Year:** Entries listing a construction year before 1800 (such as the default placeholder year 1000) are treated as invalid.
* **Physical Sanity Constraints:** Removes records with physically impossible dimensions ($TotalArea \le 0$, $TotalArea > 10000$, or $LivingArea < 16$).

### Methodology Note: TotalRooms Handling
It was considered to remove the rows with null `TotalRooms`, but we'd have around 5 000 rows less to train the models and so the solution we found was to calculate the median of it. This median imputation preserves training capacity without introducing statistical skew.

In [46]:
df_clean = df.copy()

# Normalize column names 
df_clean.columns = [c.strip() for c in df_clean.columns]

# Filter dataset to PORTO district
if "District" in df_clean.columns:
    df_clean = df_clean[
        df_clean["District"].str.lower().str.strip() == "porto"
    ]

# only apartments and houses are considered residential property types
if "Type" in df_clean.columns:
    df_clean = df_clean[
        df_clean["Type"].str.lower().isin(["apartment", "house"])
    ]

# Price must be numeric
if "Price" in df_clean.columns:
    df_clean["Price"] = pd.to_numeric(df_clean["Price"], errors="coerce")

# treat EnergyCertificate values "NC" and "No Certificate" as missing
if "EnergyCertificate" in df_clean.columns:
    df_clean["EnergyCertificate"] = df_clean["EnergyCertificate"].replace(
        {"NC": np.nan, "No Certificate": np.nan, "No certificate": np.nan}
    )

# remove rows missing Price, LivingArea and TotalArea
essential_cols = [c for c in ["Price", "LivingArea", "TotalArea"] if c in df_clean.columns]
df_clean = df_clean.dropna(subset=essential_cols)

# removing extreme outliers that may heavily distort predictions
if "Price" in df_clean.columns:
    # Raised to 1 000 000 for Porto to accommodate its higher average property pricing
    df_clean = df_clean[df_clean["Price"] <= 1000000]
    df_clean = df_clean[df_clean["Price"] >= 30000]

if "TotalArea" in df_clean.columns:
    df_clean = df_clean[df_clean["TotalArea"] > 0]
    df_clean = df_clean[df_clean["TotalArea"] <= 10000]

if "LivingArea" in df_clean.columns:
    df_clean = df_clean[df_clean["LivingArea"] >= 16]  # Removes physically unrealistic small spaces

if "TotalRooms" in df_clean.columns:
    df_clean["TotalRooms"] = pd.to_numeric(df_clean["TotalRooms"], errors="coerce")
    df_clean = df_clean[
        df_clean["TotalRooms"].isna() | (df_clean["TotalRooms"] >= 1)
    ]
    
    # impute missing TotalRooms using the median
    median_rooms = int(df_clean["TotalRooms"].median())
    df_clean["TotalRooms"] = df_clean["TotalRooms"].fillna(median_rooms)

#bathrooms
if "NumberOfBathrooms" in df_clean.columns:
    df_clean["NumberOfBathrooms"] = pd.to_numeric(
        df_clean["NumberOfBathrooms"], errors="coerce"
    )
    df_clean = df_clean.dropna(subset=["NumberOfBathrooms"])

# final dataset shape
print("Cleaned dataset shape:", df_clean.shape)

Cleaned dataset shape: (14041, 25)


While dropping rows with missing TotalRooms values was considered, doing so would result in the loss of approximately 5 000 observations. To maximize data retention and preserve the model's training capacity, we opted for Median Imputation. Filling missing entries with the median ($3$ rooms) maintains a realistic baseline for standard residential properties without introducing the statistical skew that an average (mean) would cause.

In [49]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14041 entries, 1618 to 135500
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Price                  14041 non-null  float64
 1   District               14041 non-null  object 
 2   City                   14041 non-null  object 
 3   Town                   14041 non-null  object 
 4   Type                   14041 non-null  object 
 5   EnergyCertificate      9319 non-null   object 
 6   GrossArea              3246 non-null   float64
 7   TotalArea              14041 non-null  float64
 8   Parking                14037 non-null  float64
 9   HasParking             6237 non-null   object 
 10  Floor                  4240 non-null   object 
 11  ConstructionYear       9429 non-null   float64
 12  EnergyEfficiencyLevel  7804 non-null   object 
 13  PublishDate            4377 non-null   object 
 14  Garage                 7804 non-null   object 
 15  Ele

In [51]:
df_clean.isna().mean().sort_values(ascending=False).head(15)

ConservationStatus       0.807777
LotSize                  0.792109
GrossArea                0.768820
BuiltArea                0.754362
Floor                    0.698027
PublishDate              0.688270
NumberOfWC               0.571825
HasParking               0.555801
EnergyEfficiencyLevel    0.444199
NumberOfBedrooms         0.444199
Garage                   0.444199
ElectricCarsCharging     0.444199
EnergyCertificate        0.336301
ConstructionYear         0.328467
Parking                  0.000285
dtype: float64

In [53]:
df_clean.describe(include="all").transpose().head(20)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Price,14041.0,NaN,NaN,NaN,332507.95417,172323.240737,30000.0,210000.0,305900.0,399000.0,1000000.0
District,14041,1,Porto,14041,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,14041,18,Porto,4164,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Town,14041,229,Paranhos,2101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Type,14041,2,Apartment,9877,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EnergyCertificate,9319,9,A,1855,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GrossArea,3246.0,NaN,NaN,NaN,233.951325,2599.815414,0.0,101.0,138.0,205.75,145800.0
TotalArea,14041.0,NaN,NaN,NaN,230.169931,553.399044,2.0,87.0,112.0,170.0,10000.0
Parking,14037.0,NaN,NaN,NaN,0.880673,0.767367,0.0,0.0,1.0,1.0,3.0
HasParking,6237,2,True,4905,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data Splitting and Export
To maintain behavioral consistency during model training, we isolate the master dataset into specialized subsets for **Apartments** and **Houses** and export them to clean CSV files.

In [78]:
# separate datasets based on property layout dynamics
df_apartments = df_clean[df_clean["Type"].str.lower() == "apartment"].copy()
df_houses = df_clean[df_clean["Type"].str.lower() == "house"].copy()

print(f"Final clean Apartments sample size: {df_apartments.shape[0]} rows")
print(f"Final clean Houses sample size: {df_houses.shape[0]} rows")

# save cleaned datar ready for the machine learning training notebook
df_apartments.to_csv("../data/processed/porto_apartments_clean.csv", index=False)
df_houses.to_csv("../data/processed/porto_houses_clean.csv", index=False)

print("processed data files successfully generated: 'porto_apartments_clean.csv' and 'porto_houses_clean.csv'")

Final clean Apartments sample size: 9877 rows
Final clean Houses sample size: 4164 rows
Processed data files successfully generated: 'porto_apartments_clean.csv' and 'porto_houses_clean.csv'
